# Matrix Factorization for MovieLens-1M

Having established a popularity baseline in the previous notebook, we now build
our first *personalized* model: matrix factorization (MF). The idea is to learn
a compact latent vector for each user and each movie, so that a predicted
rating is the dot product of the two, compressing the sparse rating matrix
into dense representations that generalize to unseen user-movie pairs.

We train this model to predict ratings (minimizing mean squared error) and then
evaluate it against the popularity baseline using the ranking metrics from
notebook 01. As we will see, the result is instructive: despite training
cleanly, the model *underperforms* the popularity baseline on top-k ranking,
a discrepancy we diagnose in detail, and which motivates the weighted approach
of the next notebook.

In [1]:
import numpy as np
import pandas as pd
import torch

import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))   # make src/ importable (must precede src imports)

from src.data import load_ratings, time_split, build_id_maps
from src.model import MatrixFactorization
from src.metrics import evaluate

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

## Data and Setup

We reuse the time-based train/test split and the contiguous ID mappings
developed in notebook 01, now packaged in `src/data.py`. Recall that the split
is chronological (earliest 80% for training, latest 20% for testing) to avoid
leakage, and that we restrict evaluation to *warm* users who appear in the
training set.

In [2]:
ratings = load_ratings()
train, test_warm = time_split(ratings)
user_to_idx, movie_to_idx = build_id_maps(train)
n_users, n_movies = len(user_to_idx), len(movie_to_idx)

print(f"Users: {n_users}, Movies: {n_movies}")
print(f"Train interactions: {len(train)}, Warm test interactions: {len(test_warm)}")

Users: 5400, Movies: 3662
Train interactions: 800167, Warm test interactions: 104540


## The Model

Matrix factorization represents each user $u$ and each movie $m$ as a
$k$-dimensional latent vector, $p_u$ and $q_m$. The predicted rating is their
dot product, plus bias terms that absorb systematic effects:

$$\hat{r}_{um} = \mu + b_u + b_m + p_u \cdot q_m$$

Here $\mu$ is the global average rating, $b_u$ captures how generously a user
rates overall, and $b_m$ captures how highly a movie is rated overall, leaving
the dot product $p_u \cdot q_m$ to model the personalized interaction between a
user's tastes and a movie's traits. All of these are learned jointly from the
observed ratings.

The intuition for why this generalizes: because $k$ is small (here 50) relative
to the number of movies, each vector must explain a user's ratings across many
movies using only a few numbers. This forces the vectors to capture shared
structure rather than memorize individual ratings, so the learned
representations produce sensible predictions even for unseen user-movie pairs.

In [3]:
from src.model import MatrixFactorization

## Training

We train the model to predict the observed ratings by minimizing mean squared
error between the predicted and actual rating:

$$\text{MSE} = \frac{1}{N} \sum_{(u,m)} \left( r_{um} - \hat{r}_{um} \right)^2$$

The sum runs only over *observed* ratings; the empty cells of the matrix are
absent from the loss. Training proceeds by stochastic gradient descent in
mini-batches: for each batch we compute predictions, measure the loss,
backpropagate to obtain gradients, and update the parameters. We note here,
and return to it later, that MSE optimizes the model to predict rating
*values* accurately, which is not the same objective as ranking the right
movies highly.

In [4]:
# tensors for training (dense indices + float ratings)
train_users = torch.tensor(train["user_id"].map(user_to_idx).values, dtype=torch.long)
train_movies = torch.tensor(train["movie_id"].map(movie_to_idx).values, dtype=torch.long)
train_ratings = torch.tensor(train["rating"].values, dtype=torch.float)

model = MatrixFactorization(n_users, n_movies, k=50)
loss_fn = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-5)

n_epochs = 5
batch_size = 1024
n_train = len(train_ratings)

for epoch in range(n_epochs):
    perm = torch.randperm(n_train)               # shuffle each epoch
    total_loss = 0.0
    for i in range(0, n_train, batch_size):
        idx = perm[i:i + batch_size]
        preds = model(train_users[idx], train_movies[idx])   # 1. forward
        loss = loss_fn(preds, train_ratings[idx])            # 2. loss
        optimizer.zero_grad()                                # 3. zero grads
        loss.backward()                                      # 4. backward
        optimizer.step()                                     # 5. step
        total_loss += loss.item() * len(idx)
    print(f"Epoch {epoch+1}: MSE = {total_loss / n_train:.4f}")

Epoch 1: MSE = 20.8302
Epoch 2: MSE = 2.7189
Epoch 3: MSE = 1.1796
Epoch 4: MSE = 0.8486
Epoch 5: MSE = 0.7382


## Evaluation: Comparing Against the Baseline

To evaluate, we generate a ranked top-k list for each warm user by scoring all
movies and keeping the highest-predicted ones the user has not already seen.
We then compute the same ranking metrics from notebook 01, and compare directly
against the popularity baseline.

In [5]:
# precompute each user's seen movies (dense indices) for filtering
seen_by_user = train.groupby(train["user_id"].map(user_to_idx))["movie_id"].apply(
    lambda s: set(s.map(movie_to_idx))
).to_dict()

idx_to_movie = {i: mid for mid, i in movie_to_idx.items()}
all_movie_idx = torch.arange(n_movies)

def recommend(user_id, k=10):
    """Top-k recommended original movie IDs for a user, excluding seen items."""
    user_idx = user_to_idx[user_id]
    with torch.no_grad():
        users = torch.full((n_movies,), user_idx, dtype=torch.long)
        scores = model(users, all_movie_idx)
    ranked = torch.argsort(scores, descending=True).tolist()
    seen = seen_by_user.get(user_idx, set())
    return [idx_to_movie[i] for i in ranked if i not in seen][:k]

In [ ]:
from src.baseline import recommend_popular

warm_user_ids = list(test_warm["user_id"].unique())

# ground truth: movies each user rated >= 4 in the test period
relevant_ratings = test_warm[test_warm["rating"] >= 4]
relevant_by_user = relevant_ratings.groupby("user_id")["movie_id"].apply(set).to_dict()

# matrix factorization recommendations
recs_mf = {uid: recommend(uid, k=10) for uid in warm_user_ids}
results_mf = evaluate(recs_mf, relevant_by_user, k=10)

# popularity baseline recommendations (from notebook 01)
recs_pop = recommend_popular(train, warm_user_ids, k=10)
results_pop = evaluate(recs_pop, relevant_by_user, k=10)

print("Popularity baseline:", {m: round(v, 4) for m, v in results_pop.items()})
print("Matrix factorization:", {m: round(v, 4) for m, v in results_mf.items()})

Popularity baseline: {'precision': 0.1959, 'recall': 0.0493, 'ndcg': 0.2145}
Matrix factorization: {'precision': 0.0137, 'recall': 0.0035, 'ndcg': 0.0137}


## Why Does the Personalized Model Lose?

This result is surprising: the matrix factorization model trained cleanly
(MSE fell to 0.74), yet it is badly outperformed by a non-personalized
baseline. Two reasons explain the gap.

### Reason 1: Objective mismatch

We trained the model to minimize MSE, which rewards predicting rating *values*
accurately. But our evaluation measures *ranking*, whether the right movies
appear at the top of the list. These are different objectives. A model can
predict ratings well on average while still ordering its top recommendations
poorly, because MSE treats every observed rating equally and never directly
optimizes the order of the top-k items.

### Reason 2: Popularity bias in the evaluation

Precision@10 asks how many of our 10 recommended movies the user actually rated
$\geq 4$ in the test set. The difficulty is that each user's test ratings are
dominated by *popular* movies. Rather than assume this, we verify it directly
by inspecting which highly-rated movies appear most often in the test set:

In [7]:
# load movie titles for readability
movies = pd.read_csv(
    "../data/ml-1m/movies.dat",
    sep="::", names=["movie_id", "title", "genres"],
    engine="python", encoding="latin-1",
)
title_of = dict(zip(movies["movie_id"], movies["title"]))

# most frequently liked (>=4) movies in the test set
top_liked = test_warm[test_warm["rating"] >= 4]["movie_id"].value_counts().head(15)
print("Most-rated (>=4) movies in the test set:\n")
for mid, count in top_liked.items():
    print(f"  {count:4d}  {title_of.get(mid, '???')}")

Most-rated (>=4) movies in the test set:

   261  Almost Famous (2000)
   230  Gladiator (2000)
   196  Wonder Boys (2000)
   184  Best in Show (2000)
   164  Chicken Run (2000)
   157  Sixth Sense, The (1999)
   150  Godfather, The (1972)
   149  High Fidelity (2000)
   146  Star Wars: Episode IV - A New Hope (1977)
   144  Meet the Parents (2000)
   144  Erin Brockovich (2000)
   141  Requiem for a Dream (2000)
   137  Shawshank Redemption, The (1994)
   134  Silence of the Lambs, The (1991)
   134  X-Men (2000)


The most-rated movies in the test set are widely-known, recently-released
films (Gladiator, The Sixth Sense, X-Men, and other 2000-era hits). This
reflects both their broad appeal and the time-based split, which places recent
releases in the test window. The popularity baseline recommends exactly these
movies to every user, so its recommendations overlap heavily with what users
rated. The personalized model instead tailors recommendations to individual
taste, often surfacing well-matched but less popular movies. Even when these
are good recommendations, they rarely coincide with the popular titles filling
the user's sparse test set, so they score as misses. The metric cannot
distinguish a poor recommendation from a good one that simply is not among the
popular movies the user happened to rate.

## Conclusion

Our first personalized model underperforms a simple popularity baseline, but
the reason is subtle and instructive. The model does learn, it trained cleanly
and produces personalized recommendations, but two factors work against it: it
was trained to predict ratings rather than to rank, and our evaluation is
biased toward popular movies that dominate the sparse test set.

This raises a deeper question: is the model genuinely weak, or is our
*evaluation* simply ill-suited to measuring personalization? Top-k precision
against held-out ratings rewards recommending popular items, which a
non-personalized baseline does by construction. A fairer test would isolate
whether the model can distinguish movies a user likes from movies they do not,
independent of popularity.

In the next notebook we pursue both threads. We address the objective mismatch
by switching to weighted matrix factorization, which trains on an implicit
preference signal rather than rating values. And we address the evaluation bias
by introducing a leave-one-out protocol, which scores each user's held-out item
against random alternatives, a measure far less dominated by popularity. Under
that fairer evaluation, the personalized model demonstrates the value the top-k
metric could not reveal.